In [7]:
import duckdb

### 1. Download Data

Done using the script/download_trains_data.rb

### 2. Put stations data into stations table in DuckDB. This changes rarely, so we treat it as a almost constant file.

In [18]:
with duckdb.connect("data/duckdb_trains.db") as db:
    db.sql("""
        CREATE TABLE IF NOT EXISTS stations AS
        FROM "data/stations-2023-09.csv"
    """)
    print("stations imported to data/duckdb_trains.db")

    display(db.sql("FROM stations SELECT COUNT(*) AS stations_count"))

stations imported to data/duckdb_trains.db


┌────────────────┐
│ stations_count │
│     int64      │
├────────────────┤
│            591 │
└────────────────┘

### 3. Based on DuckDB tutorial, create tables distances and distances_long. We treat this similarly to stations table.

In [9]:
!head -n 9 data/tariff-distances-2022-01.csv | cut -d, -f1-9

Station,AC,AH,AHP,AHPR,AHZ,AKL,AKM,ALM
AC,XXX,82,83,85,90,71,188,32
AH,82,XXX,1,3,8,77,153,98
AHP,83,1,XXX,2,9,78,152,99
AHPR,85,3,2,XXX,11,80,150,101
AHZ,90,8,9,11,XXX,69,161,106
AKL,71,77,78,80,69,XXX,211,96
AKM,188,153,152,150,161,211,XXX,158
ALM,32,98,99,101,106,96,158,XXX


In [21]:
with duckdb.connect("data/duckdb_trains.db") as db:
    db.sql("""
        CREATE TABLE IF NOT EXISTS distances AS
        FROM read_csv('data/tariff-distances-2022-01.csv', nullstr = 'XXX')
    """)

    display(db.sql("FROM (DESCRIBE distances) LIMIT 5;"))
    display(db.sql("FROM distances SELECT COUNT(*) AS distances_count"))
    display(db.sql(
        """SELECT #1, #2, #3, #4, #5, #6, #7, #8, #9
            FROM distances
            LIMIT 8;
        """))



┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ Station     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ AC          │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ AH          │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ AHP         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ AHPR        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

┌─────────────────┐
│ distances_count │
│      int64      │
├─────────────────┤
│             399 │
└─────────────────┘

┌─────────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┐
│ Station │  AC   │  AH   │  AHP  │ AHPR  │  AHZ  │  AKL  │  AKM  │  ALM  │
│ varchar │ int64 │ int64 │ int64 │ int64 │ int64 │ int64 │ int64 │ int64 │
├─────────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┤
│ AC      │  NULL │    82 │    83 │    85 │    90 │    71 │   188 │    32 │
│ AH      │    82 │  NULL │     1 │     3 │     8 │    77 │   153 │    98 │
│ AHP     │    83 │     1 │  NULL │     2 │     9 │    78 │   152 │    99 │
│ AHPR    │    85 │     3 │     2 │  NULL │    11 │    80 │   150 │   101 │
│ AHZ     │    90 │     8 │     9 │    11 │  NULL │    69 │   161 │   106 │
│ AKL     │    71 │    77 │    78 │    80 │    69 │  NULL │   211 │    96 │
│ AKM     │   188 │   153 │   152 │   150 │   161 │   211 │  NULL │   158 │
│ ALM     │    32 │    98 │    99 │   101 │   106 │    96 │   158 │  NULL │
└─────────┴───────┴───────┴───────┴───────┴───────┴───────┴───────┴───────┘

In [26]:
with duckdb.connect("data/duckdb_trains.db") as db:
    display(db.sql("""
        CREATE TABLE IF NOT EXISTS distances_long AS
            UNPIVOT distances
            ON COLUMNS (* EXCLUDE station)
            INTO NAME other_station VALUE distance;
    """))
    display(db.sql("""
        SELECT station, other_station, distance
            FROM distances_long
            LIMIT 5;
    """))
    display(db.sql("""
        SELECT
            s1.name_long AS station1,
            s2.name_long AS station2,
            distances_long.distance
        FROM distances_long
        JOIN stations s1 ON distances_long.station = s1.code
        JOIN stations s2 ON distances_long.other_station = s2.code
        WHERE s1.country = 'NL'
          AND s2.country = 'NL'
          AND station < other_station
        ORDER BY distance DESC
        LIMIT 5;
    """))





None

┌─────────┬───────────────┬──────────┐
│ Station │ other_station │ distance │
│ varchar │    varchar    │  int64   │
├─────────┼───────────────┼──────────┤
│ AC      │ AH            │       82 │
│ AC      │ AHP           │       83 │
│ AC      │ AHPR          │       85 │
│ AC      │ AHZ           │       90 │
│ AC      │ AKL           │       71 │
└─────────┴───────────────┴──────────┘

┌──────────────────┬────────────────────┬──────────┐
│     station1     │      station2      │ distance │
│     varchar      │      varchar       │  int64   │
├──────────────────┼────────────────────┼──────────┤
│ Eemshaven        │ Vlissingen         │      426 │
│ Eemshaven        │ Vlissingen Souburg │      425 │
│ Bad Nieuweschans │ Vlissingen         │      425 │
│ Bad Nieuweschans │ Vlissingen Souburg │      424 │
│ Stavoren         │ Vlissingen         │      421 │
└──────────────────┴────────────────────┴──────────┘

### 4. Put train disruptions into disruptions table in the Postgres database. We expect this data to change regularly, and thus treat it as a typical OLTP table.

In [36]:
conn_string = "host=localhost user=postgres password=postgres dbname=postgres"

# Create read-write connection to Postgres
with duckdb.connect("data/duckdb_trains.db") as db:
    db.sql(f"""
    ATTACH IF NOT EXISTS '{conn_string}' AS postgres_db (TYPE postgres);
    """)

    # load 2011-2023 disruptions into Postgres
    db.sql("""
        CREATE OR REPLACE TABLE postgres_db.disruptions AS
            FROM read_csv('data/disruptions-*.csv')
    """)


    display(db.sql("""
        SELECT
            YEAR(start_time) AS year,
            COUNT(*) AS num_disruptions
        FROM postgres_db.disruptions
        WHERE YEAR(start_time) BETWEEN 2011 AND 2023
        GROUP BY YEAR(start_time)
        ORDER BY year
    """))



┌───────┬─────────────────┐
│ year  │ num_disruptions │
│ int64 │      int64      │
├───────┼─────────────────┤
│  2011 │            1846 │
│  2012 │            2074 │
│  2013 │            2312 │
│  2014 │            2484 │
│  2015 │            2947 │
│  2016 │            3031 │
│  2017 │            4085 │
│  2018 │            5190 │
│  2019 │            5940 │
│  2020 │            4450 │
│  2021 │            4874 │
│  2022 │            5499 │
│  2023 │            5168 │
├───────┴─────────────────┤
│ 13 rows       2 columns │
└─────────────────────────┘

### 5. Transform train services CSV files into a single Parquet file. Make table services from it. We treat this as a big data batch input, created rarely but regularly for analytics purposes.
> We treat this as a big data batch input, created rarely but regularly for analytics purposes.

We want to store it in duckdb.

In [38]:
# 5.1.  lets transform services 2019-2025 CSVs into Parquet file
with duckdb.connect("data/duckdb_trains.db") as db:
    db.sql(f"""
        COPY (
            SELECT * FROM read_csv('data/services-20??.csv', union_by_name=True)
        ) TO 'data/services_2019_2025.parquet' (FORMAT PARQUET);
    """)

    print(db.sql("SELECT count(*) FROM 'data/services_2019_2025.parquet'").fetchall())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[(152176324,)]


In [44]:
with duckdb.connect("data/duckdb_trains.db") as db:
    display(db.sql("SELECT * FROM 'data/services_2019_2025.parquet' LIMIT 5;"))
    display(db.sql("""SELECT YEAR("Service:Date"), count(*) FROM 'data/services_2019_2025.parquet' GROUP BY YEAR("Service:Date")""").fetchall())

┌────────────────┬──────────────┬──────────────┬─────────────────┬──────────────────────┬──────────────────────────────┬──────────────────────────┬───────────────────────┬─────────────┬───────────────────┬────────────────────┬─────────────────────┬────────────────────┬────────────────────────┬─────────────────────┬──────────────────────┬──────────────────────────┬──────────────────────┬───────────────────────┬──────────────────────┐
│ Service:RDT-ID │ Service:Date │ Service:Type │ Service:Company │ Service:Train number │ Service:Completely cancelled │ Service:Partly cancelled │ Service:Maximum delay │ Stop:RDT-ID │ Stop:Station code │ Stop:Station name  │  Stop:Arrival time  │ Stop:Arrival delay │ Stop:Arrival cancelled │ Stop:Departure time │ Stop:Departure delay │ Stop:Departure cancelled │ Stop:Platform change │ Stop:Planned platform │ Stop:Actual platform │
│     int64      │     date     │   varchar    │     varchar     │        int64         │           boolean            │      

[(2019, 20773804),
 (2020, 22235137),
 (2021, 21818011),
 (2022, 22025754),
 (2023, 21239393),
 (2024, 21857914),
 (2025, 22226311)]

In [ ]:
# 5.2. Create `services` table in duckdb from the parquet file.

# TODO !!!!